# Episode Studio — WanGP to final video

Run all cells to launch the end-to-end browser studio. Upload `scene-plan.json` and `voiceover-source.txt`; the selected model generates the scenes, Kokoro creates narration, Whisper timestamps it, and HyperFrames renders the final MP4.


## 1. Choose the runtime mode and verify hardware

In [ ]:
import os, subprocess
from pathlib import Path

# image-z: Z-Image Turbo stills (T4-friendly)
# image-ltx: LTX-2.5 stills (high-memory runtime)
# video-ltx: LTX-2.5 clips (high-memory runtime)
STUDIO_MODE = 'image-z'
PERSIST_TO_DRIVE = False
ADVANCED_ALLOW_LOW_MEMORY_LTX = False

if STUDIO_MODE not in {'image-z', 'image-ltx', 'video-ltx'}:
    raise ValueError('STUDIO_MODE must be image-z, image-ltx, or video-ltx')
subprocess.run(['nvidia-smi'], check=True)
gpu_line = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True).strip().splitlines()[0]
GPU_NAME, memory_mib = [part.strip() for part in gpu_line.rsplit(',', 1)]
GPU_VRAM_GB = float(memory_mib) / 1024
SYSTEM_RAM_GB = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
print(f'Mode: {STUDIO_MODE}')
print(f'GPU: {GPU_NAME} · {GPU_VRAM_GB:.1f} GiB VRAM')
print(f'System RAM: {SYSTEM_RAM_GB:.1f} GiB')
if STUDIO_MODE.endswith('ltx') and not ADVANCED_ALLOW_LOW_MEMORY_LTX and (GPU_VRAM_GB < 24 or SYSTEM_RAM_GB < 60):
    raise RuntimeError('LTX-2.5 defaults to WanGP profile 1 and needs about 24 GiB VRAM plus 64 GB system RAM. Select a larger/high-RAM runtime, use image-z, or explicitly enable the advanced lower-memory override.')


## 2. Download or update Episode Studio

In [ ]:
import subprocess

WAN2GP_ROOT = Path('/content/wan2gp').resolve()
REPOSITORY = 'https://github.com/hoangthvn2201/wan2gp-optimized.git'
BRANCH = 'main'
if (WAN2GP_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY, str(WAN2GP_ROOT)], check=True)
subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
required = [WAN2GP_ROOT / 'wan2gp_server/production.py', WAN2GP_ROOT / 'wan2gp_server/static/episode-index.html']
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError(f'The selected branch does not contain Episode Studio: {missing}')
print('Repository ready:', WAN2GP_ROOT)


## 3. Install system and Python dependencies

In [ ]:
import os, subprocess, sys

install_env = os.environ.copy()
install_env['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=install_env)
def apt_choice(*candidates):
    for candidate in candidates:
        probe = subprocess.run(['apt-cache', 'show', '--no-all-versions', candidate], text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        if probe.returncode == 0 and probe.stdout.strip():
            return candidate
    raise RuntimeError(f'No compatible apt package found for: {candidates}')
browser_packages = ['ca-certificates', 'fonts-liberation', 'libcairo2', 'libdbus-1-3', 'libdrm2', 'libexpat1', 'libfontconfig1', 'libgbm1', 'libnspr4', 'libnss3', 'libpango-1.0-0', 'libpangocairo-1.0-0', 'libx11-6', 'libx11-xcb1', 'libxcb1', 'libxcomposite1', 'libxcursor1', 'libxdamage1', 'libxext6', 'libxfixes3', 'libxi6', 'libxkbcommon0', 'libxrandr2', 'libxrender1', 'libxshmfence1', 'libxss1', 'libxtst6', 'xdg-utils']
browser_packages += [apt_choice('libasound2t64', 'libasound2'), apt_choice('libatk-bridge2.0-0t64', 'libatk-bridge2.0-0'), apt_choice('libatk1.0-0t64', 'libatk1.0-0'), apt_choice('libatspi2.0-0t64', 'libatspi2.0-0'), apt_choice('libcups2t64', 'libcups2'), apt_choice('libgtk-3-0t64', 'libgtk-3-0')]
subprocess.run(['sudo', 'apt-get', 'install', '-y', '--no-install-recommends', 'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2', 'espeak-ng', *browser_packages], check=True, env=install_env)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', 'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'xformers'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WAN2GP_ROOT / 'requirements.txt'), '-r', str(WAN2GP_ROOT / 'wan2gp_server/requirements.txt')], check=True)
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
if target.exists():
    source = target.read_text()
    if "matplotlib.use('TkAgg')" in source:
        target.write_text(source.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')", 1))
print('Python environment ready.')


## 4. Install and warm HyperFrames, Kokoro, and transcription

In [ ]:
import os, platform, shutil, subprocess

HYPERFRAMES_VERSION = '0.8.3'
NODE_MIN_MAJOR = 22
node_version = subprocess.check_output(['node', '--version'], text=True).strip()
node_major = int(node_version.lstrip('v').split('.', 1)[0])
if node_major < NODE_MIN_MAJOR:
    print(f'HyperFrames needs Node >= {NODE_MIN_MAJOR}; upgrading Colab from {node_version}...')
    subprocess.run(['npm', 'install', '--global', 'n'], check=True)
    subprocess.run(['n', str(NODE_MIN_MAJOR)], check=True)
node_version = subprocess.check_output(['node', '--version'], text=True).strip()
if int(node_version.lstrip('v').split('.', 1)[0]) < NODE_MIN_MAJOR:
    raise RuntimeError(f'Node upgrade did not take effect; found {node_version}, need >= {NODE_MIN_MAJOR}. Restart the runtime and rerun the notebook.')
print('Node:', node_version)
subprocess.run(['npm', 'install', '--global', f'hyperframes@{HYPERFRAMES_VERSION}'], check=True)

def visible_run(command):
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout)
    return result

browser_result = visible_run(['hyperframes', 'browser', 'ensure'])
if browser_result.returncode != 0:
    # A managed Chrome download can occasionally be unavailable from a Colab VM.
    # HyperFrames can render through regular system Chrome's screenshot path instead.
    browser_path = next((path for name in ('google-chrome', 'google-chrome-stable', 'chromium') if (path := shutil.which(name))), None)
    if browser_path is None and platform.machine().lower() in {'x86_64', 'amd64'}:
        chrome_deb = Path('/tmp/google-chrome-stable_current_amd64.deb')
        subprocess.run(['wget', '-q', '-O', str(chrome_deb), 'https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb'], check=True)
        subprocess.run(['sudo', 'apt-get', 'install', '-y', str(chrome_deb)], check=True)
        browser_path = shutil.which('google-chrome') or shutil.which('google-chrome-stable')
    if browser_path is None:
        raise RuntimeError('HyperFrames could not download its managed browser and no system Chrome/Chromium is available. See the full browser error above.')
    os.environ['HYPERFRAMES_BROWSER_PATH'] = browser_path
    print('Using system browser fallback:', browser_path)
    subprocess.run(['hyperframes', 'browser', 'path'], check=True)
browser_path = subprocess.check_output(['hyperframes', 'browser', 'path'], text=True).strip()
browser_smoke = subprocess.run([browser_path, '--headless', '--no-sandbox', '--disable-gpu', '--disable-dev-shm-usage', '--dump-dom', 'about:blank'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60)
if browser_smoke.returncode != 0:
    raise RuntimeError(f'HyperFrames browser was downloaded but cannot launch.\n{browser_smoke.stdout[-6000:]}')
print('Browser launch verified:', browser_path)
warm_root = Path('/content/hyperframes-warmup')
warm_root.mkdir(parents=True, exist_ok=True)
warm_audio = warm_root / 'speech.wav'
subprocess.run(['hyperframes', 'tts', 'Episode Studio is ready.', '--voice', 'am_adam', '--output', str(warm_audio)], check=True)
subprocess.run(['hyperframes', 'transcribe', str(warm_audio), '--dir', str(warm_root), '--engine', 'whisper', '--model', 'small.en'], check=True)
print('HyperFrames, Kokoro, browser, and Whisper are warm.')


## 5. Configure project persistence and start the server

In [ ]:
import os, subprocess, sys, time
import requests

if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/episode-studio-data')
else:
    DATA_DIR = Path('/content/episode-studio-data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path('/content/episode-studio-outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PORT = 8000
MEMORY_PROFILE = 5 if STUDIO_MODE == 'image-z' else (3.5 if ADVANCED_ALLOW_LOW_MEMORY_LTX else 1)
is_ltx = STUDIO_MODE.endswith('ltx')
SERVER_ENV = {
    'WAN2GP_ROOT': str(WAN2GP_ROOT),
    'WAN2GP_SERVER_PORT': str(PORT),
    'WAN2GP_SERVER_DATA_DIR': str(DATA_DIR),
    'WAN2GP_SERVER_OUTPUT_DIR': str(OUTPUT_DIR),
    'WAN2GP_CLI_ARGS': f'--profile {MEMORY_PROFILE} --attention sdpa',
    'WAN2GP_STUDIO_MODE': STUDIO_MODE,
    'WAN2GP_HYPERFRAMES_BIN': 'hyperframes',
    'WAN2GP_SERVER_T2I_MODEL': 'ltx25-distilled-image' if is_ltx else 'z-image-turbo',
    'WAN2GP_SERVER_T2V_MODEL': 'ltx25-distilled' if is_ltx else 'wan21-t2v-1.3b',
    'WAN2GP_SERVER_I2V_MODEL': 'ltx25-distilled-i2v' if is_ltx else 'wan21-fun-inp-1.3b',
    'PYTHONUNBUFFERED': '1',
}
LOG_PATH = Path('/content/episode-studio-server.log')
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate(); server_proc.wait(timeout=20)
server_env = os.environ.copy(); server_env.update(SERVER_ENV)
server_log = open(LOG_PATH, 'w', buffering=1)
server_proc = subprocess.Popen([sys.executable, '-u', '-m', 'wan2gp_server'], cwd=str(WAN2GP_ROOT), env=server_env, stdout=server_log, stderr=subprocess.STDOUT)
BASE_URL = f'http://127.0.0.1:{PORT}'
for _ in range(90):
    if server_proc.poll() is not None:
        raise RuntimeError(f'Server exited with {server_proc.returncode}:\n{LOG_PATH.read_text(errors="replace")[-12000:]}')
    try:
        response = requests.get(f'{BASE_URL}/health', timeout=2)
        if response.ok:
            print('Server online:', response.json()); break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError(f'Server did not start. Log:\n{LOG_PATH.read_text(errors="replace")[-12000:]}')


## 6. Download and preload the selected generation model

In [ ]:
import sys, time
import requests
if str(WAN2GP_ROOT) not in sys.path:
    sys.path.insert(0, str(WAN2GP_ROOT))
from wan2gp_server.client import Wan2GPServerClient
client = Wan2GPServerClient(BASE_URL, timeout=120)
PRELOAD_MODEL = 'z-image-turbo' if STUDIO_MODE == 'image-z' else 'ltx25-distilled'
existing = next((item for item in client.jobs(limit=50) if item.get('task') == 'preload' and item.get('model') == PRELOAD_MODEL and item.get('status') in {'queued', 'running'}), None)
if existing:
    job = existing
    print(f"Resuming preload {PRELOAD_MODEL} as job {job['id']}.")
else:
    queued = client.preload([PRELOAD_MODEL], wait=False)
    if not queued:
        raise RuntimeError('Preload endpoint returned no job')
    job = queued[0]
job_id = job['id']; last = ''; disconnected_at = None
print(f'Preloading {PRELOAD_MODEL} as job {job_id}. First run downloads checkpoint files.')
deadline = time.monotonic() + 4 * 3600
while time.monotonic() < deadline:
    exit_code = server_proc.poll()
    if exit_code is not None:
        hint = '\nThe process was killed by Colab, usually due to system-RAM exhaustion. Select a High-RAM runtime and rerun all cells.' if exit_code in {-9, 137} else ''
        raise RuntimeError(f'Server exited with code {exit_code} during warmup.{hint}\nServer log:\n{LOG_PATH.read_text(errors="replace")[-14000:]}')
    try:
        job = client.job(job_id)
        if disconnected_at is not None:
            print('✓ Reconnected to the local server; preload is still running.')
            disconnected_at = None
    except (requests.ConnectionError, requests.Timeout) as exc:
        if disconnected_at is None:
            disconnected_at = time.monotonic()
            print(f'[reconnecting] Local server reset the connection during model load: {exc}')
        time.sleep(3)
        exit_code = server_proc.poll()
        if exit_code is not None:
            hint = '\nThe process was killed by Colab, usually due to system-RAM exhaustion. Select a High-RAM runtime and rerun all cells.' if exit_code in {-9, 137} else ''
            raise RuntimeError(f'Server exited with code {exit_code} during warmup.{hint}\nServer log:\n{LOG_PATH.read_text(errors="replace")[-14000:]}') from exc
        if time.monotonic() - disconnected_at > 300:
            raise RuntimeError(f'Local server remained unreachable for five minutes.\nServer log:\n{LOG_PATH.read_text(errors="replace")[-14000:]}') from exc
        continue
    progress = job.get('progress') or {}
    line = f"[{job['status']}] {progress.get('phase') or ''} {progress.get('percent', 0)}% {progress.get('status') or ''}".strip()
    if line != last: print(line); last = line
    if job['status'] in {'succeeded', 'failed', 'cancelled'}: break
    time.sleep(5)
else:
    raise RuntimeError('Model preload exceeded four hours')
if job['status'] != 'succeeded':
    raise RuntimeError(f"Warmup {job['status']}: {job.get('error')}\n{LOG_PATH.read_text(errors='replace')[-14000:]}")
print(f'✓ {PRELOAD_MODEL} is loaded. Episode Studio is ready.')


## 7. Open the public Episode Studio

In [ ]:
import os, queue, re, subprocess, threading
from IPython.display import HTML, display
CLOUDFLARED = Path('/content/cloudflared')
if not CLOUDFLARED.exists():
    subprocess.run(['wget', '-q', '-O', str(CLOUDFLARED), 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', str(CLOUDFLARED)], check=True)
if 'tunnel_proc' in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
tunnel_proc = subprocess.Popen([str(CLOUDFLARED), 'tunnel', '--url', BASE_URL, '--protocol', 'http2', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
urls = queue.Queue()
def watch_tunnel():
    for line in iter(tunnel_proc.stdout.readline, ''):
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match and urls.empty(): urls.put(match.group(0))
threading.Thread(target=watch_tunnel, daemon=True).start()
public_url = urls.get(timeout=90)
display(HTML(f'<div style="padding:22px;border-radius:18px;background:#0d192a;color:white;font-family:system-ui"><b style="color:#ed674f">EPISODE STUDIO IS READY</b><br><a href="{public_url}" target="_blank" style="display:inline-block;margin-top:14px;padding:12px 18px;border-radius:10px;background:#ed674f;color:white;text-decoration:none;font-weight:800">Open Episode Studio ↗</a><p style="opacity:.6;font-size:12px">Projects: {DATA_DIR}<br>Server log: {LOG_PATH}</p></div>'))
print('Studio:', public_url)


## Operations

Projects persist under `DATA_DIR`. Refreshing the browser does not cancel work. If the Colab process restarts, rerun the notebook and use **Resume pipeline** on an interrupted project. The legacy low-level Frameflow UI remains at `/frameflow`.


In [ ]:
# Optional shutdown
# if 'tunnel_proc' in globals() and tunnel_proc.poll() is None: tunnel_proc.terminate()
# if 'server_proc' in globals() and server_proc.poll() is None: server_proc.terminate()
# print('Episode Studio stopped.')
